# Tutorial 2: Quality Filtering

LLM-generated text is not always usable as training data. Common problems include:
- Empty or very short outputs
- **Label leakage** -- the generated text contains the label name itself (e.g., "This is a positive review...")
- Near-duplicate samples that inflate the dataset without adding diversity
- Samples with suspicious keyword patterns

This tutorial covers the filter pipeline shipped with `synthetictext` and shows how to compose your own.

In [1]:
import pandas as pd
from synthetictext import TaskSpec

## Simulated Raw Data

Let's create a small DataFrame that mimics typical LLM output -- including several problematic samples.

In [2]:
raw_data = pd.DataFrame({
    "id": [f"s{i}" for i in range(10)],
    "text": [
        "This laptop is absolutely amazing, the battery lasts all day and the screen is gorgeous!",
        "Terrible experience. The phone broke after two weeks and customer service was unhelpful.",
        "Label: positive. This product exceeded all expectations and I would buy it again.",  # leakage
        "",  # empty
        "Great headphones with crystal clear sound quality and comfortable fit for long sessions.",
        "This laptop is absolutely amazing, the battery lasts all day and the screen is gorgeous!",  # duplicate of row 0
        "The classification for this text is negative. Bad product.",  # leakage
        "ok",  # too short
        "Worst purchase ever. Completely waste of money, arrived damaged and smelled terrible.",
        "I love this coffee maker! Perfect temperature every time, easy to clean, highly recommend.",
    ],
    "label": [1, 0, 1, 1, 1, 1, 0, 0, 0, 1],
    "source": ["direct"] * 10,
})

print(f"Raw samples: {len(raw_data)}")
raw_data

Raw samples: 10


,id,text,label,source
0,s0,"This laptop is absolutely amazing, the battery...",1,direct
1,s1,Terrible experience. The phone broke after two...,0,direct
2,s2,Label: positive. This product exceeded all exp...,1,direct
3,s3,,1,direct
4,s4,Great headphones with crystal clear sound qual...,1,direct
5,s5,"This laptop is absolutely amazing, the battery...",1,direct
6,s6,The classification for this text is negative. ...,0,direct
7,s7,ok,0,direct
8,s8,Worst purchase ever. Completely waste of money...,0,direct
9,s9,I love this coffee maker! Perfect temperature ...,1,direct


## Individual Filters

### BasicFilter

Removes empty text, invalid labels, and text outside length bounds.

In [3]:
from synthetictext.filters.basic import BasicFilter

basic = BasicFilter(
    valid_labels=[0, 1],
    min_length=10,
    max_length=500,
)

after_basic = basic.filter(raw_data)
print(f"After BasicFilter: {len(raw_data)} -> {len(after_basic)}")
print(f"Removed: empty string (row 3), too-short 'ok' (row 7)")
after_basic

After BasicFilter: 10 -> 8
Removed: empty string (row 3), too-short 'ok' (row 7)


,id,text,label,source
0,s0,"This laptop is absolutely amazing, the battery...",1,direct
1,s1,Terrible experience. The phone broke after two...,0,direct
2,s2,Label: positive. This product exceeded all exp...,1,direct
3,s4,Great headphones with crystal clear sound qual...,1,direct
4,s5,"This laptop is absolutely amazing, the battery...",1,direct
5,s6,The classification for this text is negative. ...,0,direct
6,s8,Worst purchase ever. Completely waste of money...,0,direct
7,s9,I love this coffee maker! Perfect temperature ...,1,direct


### LeakageFilter

Detects samples where the LLM leaked the label into the text. Common patterns: "Label: positive", "classified as negative", etc.

In [4]:
from synthetictext.filters.leakage import LeakageFilter

leakage = LeakageFilter(
    task_labels={0: "negative", 1: "positive"},
    language="en",
)

after_leakage = leakage.filter(after_basic)
print(f"After LeakageFilter: {len(after_basic)} -> {len(after_leakage)}")
print(f"Removed: rows with 'Label: positive' and 'classification for this text is negative'")
after_leakage

After LeakageFilter: 8 -> 7
Removed: rows with 'Label: positive' and 'classification for this text is negative'


,id,text,label,source
0,s0,"This laptop is absolutely amazing, the battery...",1,direct
1,s1,Terrible experience. The phone broke after two...,0,direct
2,s4,Great headphones with crystal clear sound qual...,1,direct
3,s5,"This laptop is absolutely amazing, the battery...",1,direct
4,s6,The classification for this text is negative. ...,0,direct
5,s8,Worst purchase ever. Completely waste of money...,0,direct
6,s9,I love this coffee maker! Perfect temperature ...,1,direct


### EmbeddingDeduplicator

Uses sentence-transformers to compute embeddings and removes near-duplicates based on cosine similarity.

This requires the `embeddings` extra: `pip install synthetictext[embeddings]`

In [5]:
from synthetictext.filters.deduplication import EmbeddingDeduplicator

dedup = EmbeddingDeduplicator(
    threshold=0.90,  # cosine similarity above this = duplicate
    # model_name="all-MiniLM-L6-v2",  # default model; override if needed
)

after_dedup = dedup.filter(after_leakage)
print(f"After EmbeddingDeduplicator: {len(after_leakage)} -> {len(after_dedup)}")
after_dedup

C:\Users\srika\AppData\Roaming\Python\Python39\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


After EmbeddingDeduplicator: 7 -> 6


,id,text,label,source
0,s0,"This laptop is absolutely amazing, the battery...",1,direct
1,s1,Terrible experience. The phone broke after two...,0,direct
2,s4,Great headphones with crystal clear sound qual...,1,direct
3,s6,The classification for this text is negative. ...,0,direct
4,s8,Worst purchase ever. Completely waste of money...,0,direct
5,s9,I love this coffee maker! Perfect temperature ...,1,direct


### MarkerFilter

Checks whether samples of a given class contain at least one expected keyword or marker phrase. This catches cases where the LLM generates text that is topically correct but misses the distinguishing linguistic patterns.

This is optional -- only applies when `TaskSpec.marker_keywords` is set.

In [6]:
from synthetictext.filters.markers import MarkerFilter

task_with_markers = TaskSpec(
    name="Sentiment Analysis",
    labels={0: "negative", 1: "positive"},
    description="Classify product reviews as positive or negative.",
    marker_keywords={
        0: ["terrible", "worst", "broke", "waste", "damaged", "awful"],
        1: ["amazing", "love", "great", "perfect", "recommend", "excellent"],
    },
)

marker = MarkerFilter(task_with_markers)
after_markers = marker.filter(after_dedup)
print(f"After MarkerFilter: {len(after_dedup)} -> {len(after_markers)}")
after_markers

After MarkerFilter: 6 -> 5


,id,text,label,source
0,s0,"This laptop is absolutely amazing, the battery...",1,direct
1,s1,Terrible experience. The phone broke after two...,0,direct
2,s4,Great headphones with crystal clear sound qual...,1,direct
3,s8,Worst purchase ever. Completely waste of money...,0,direct
4,s9,I love this coffee maker! Perfect temperature ...,1,direct


## Default Pipeline

Instead of composing filters manually, you can use `FilterPipeline.default()` which chains: BasicFilter -> LeakageFilter -> EmbeddingDeduplicator -> MarkerFilter.

In [7]:
from synthetictext.filters.pipeline import FilterPipeline

pipeline = FilterPipeline.default(
    task_with_markers,
    min_length=10,
    max_length=500,
    use_embeddings=True,
    dedup_threshold=0.90,
)

cleaned = pipeline.run(raw_data, language="en")
print(f"\nFinal result: {len(raw_data)} -> {len(cleaned)} samples")
cleaned

C:\Users\srika\AppData\Roaming\Python\Python39\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(



Final result: 10 -> 5 samples


,id,text,label,source
0,s0,"This laptop is absolutely amazing, the battery...",1,direct
1,s1,Terrible experience. The phone broke after two...,0,direct
2,s4,Great headphones with crystal clear sound qual...,1,direct
3,s8,Worst purchase ever. Completely waste of money...,0,direct
4,s9,I love this coffee maker! Perfect temperature ...,1,direct


## Custom Pipeline

You can compose any subset of filters in any order.

In [8]:
custom_pipeline = FilterPipeline()
custom_pipeline.add(BasicFilter(valid_labels=[0, 1], min_length=20))
custom_pipeline.add(LeakageFilter(task_labels={0: "negative", 1: "positive"}))
# Skip dedup and markers for speed

result = custom_pipeline.run(raw_data)
print(f"Custom pipeline: {len(raw_data)} -> {len(result)} samples")
result

Custom pipeline: 10 -> 7 samples


,id,text,label,source
0,s0,"This laptop is absolutely amazing, the battery...",1,direct
1,s1,Terrible experience. The phone broke after two...,0,direct
2,s4,Great headphones with crystal clear sound qual...,1,direct
3,s5,"This laptop is absolutely amazing, the battery...",1,direct
4,s6,The classification for this text is negative. ...,0,direct
5,s8,Worst purchase ever. Completely waste of money...,0,direct
6,s9,I love this coffee maker! Perfect temperature ...,1,direct


## Using Filters with the Generator

When you call `generator.generate(apply_filters=True)` (the default), the default filter pipeline runs automatically. You can also pass a custom pipeline:

In [9]:
from synthetictext import SyntheticDataGenerator

generator = SyntheticDataGenerator(
    task=task_with_markers,
    llm_provider="openai",
)

# Using default filters (automatic)
# df = generator.generate(language="English", num_samples=50)

# Using a custom filter pipeline
# df = generator.generate(
#     language="English",
#     num_samples=50,
#     filter_pipeline=custom_pipeline,
# )

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

## Writing a Custom Filter

Extend `BaseFilter` to create your own filter. The only requirement is implementing the `name` property and `filter()` method.

In [ ]:
from synthetictext.filters.base import BaseFilter


class ProfanityFilter(BaseFilter):
    """Example: remove samples containing profanity."""

    def __init__(self, blocklist: list[str]):
        self._blocklist = [w.lower() for w in blocklist]

    @property
    def name(self) -> str:
        return "ProfanityFilter"

    def filter(self, df: pd.DataFrame, **kwargs) -> pd.DataFrame:
        def is_clean(text: str) -> bool:
            text_lower = text.lower()
            return not any(word in text_lower for word in self._blocklist)

        mask = df["text"].astype(str).apply(is_clean)
        return df[mask].copy().reset_index(drop=True)


# Use it in a pipeline
pipeline = FilterPipeline()
pipeline.add(BasicFilter(valid_labels=[0, 1]))
pipeline.add(ProfanityFilter(blocklist=["damn", "hell"]))

print(f"Custom filter pipeline with {len(pipeline._filters)} filters")

## Summary

- **BasicFilter**: removes empty, too-short, too-long, and invalid-label samples
- **LeakageFilter**: catches label names and classification meta-text inside generated samples
- **EmbeddingDeduplicator**: removes near-duplicate samples using cosine similarity on sentence embeddings
- **MarkerFilter**: verifies that class-specific keywords appear in the generated text
- **FilterPipeline**: composes any combination of filters in sequence
- **Custom filters**: extend `BaseFilter` to add your own logic

**Next:** [Tutorial 3: Multilingual Generation](./03_multilingual_generation.ipynb) covers generating data across multiple languages with backtranslation and pivot translation.